In [1]:
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

In [2]:
df = pd.read_csv('reddit_preprocessing.csv').dropna(subset=['clean_comment'])
df.shape

(36662, 2)

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
from imblearn.combine import SMOTEENN
from imblearn.over_sampling import ADASYN, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split


# Step 1: Function to run the experiment
def run_imbalanced_experiment(imbalance_method):
    ngram_range = (1, 3)  # Trigram setting
    max_features = 10000  # Set max_features to 10000 for TF-IDF

    # Step 4: Train-test split before vectorization and resampling
    X_train, X_test, y_train, y_test = train_test_split(
        df["clean_comment"],
        df["category"],
        test_size=0.2,
        random_state=42,
        stratify=df["category"],
    )

    # Step 2: Vectorization using TF-IDF, fit on training data only
    vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
    X_train_vec = vectorizer.fit_transform(X_train)  # Fit on training data
    X_test_vec = vectorizer.transform(X_test)  # Transform test data

    # Step 3: Handle class imbalance based on the selected method (only applied to the training set)
    if imbalance_method == "class_weights":
        # Use class_weight in Random Forest
        class_weight = "balanced"
    else:
        class_weight = None  # Do not apply class_weight if using resampling

        # Resampling Techniques (only apply to the training set)
        if imbalance_method == "oversampling":
            smote = SMOTE(random_state=42)
            X_train_vec, y_train = smote.fit_resample(X_train_vec, y_train)
        elif imbalance_method == "adasyn":
            adasyn = ADASYN(random_state=42)
            X_train_vec, y_train = adasyn.fit_resample(X_train_vec, y_train)
        elif imbalance_method == "undersampling":
            rus = RandomUnderSampler(random_state=42)
            X_train_vec, y_train = rus.fit_resample(X_train_vec, y_train)
        elif imbalance_method == "smote_enn":
            smote_enn = SMOTEENN(random_state=42)
            X_train_vec, y_train = smote_enn.fit_resample(X_train_vec, y_train)

    # Step 5: Define and train a Random Forest model
    n_estimators = 200
    max_depth = 15

    # Initialize and train the model
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
        class_weight=class_weight,
    )
    model.fit(X_train_vec, y_train)

    # Step 6: Make predictions and evaluate metrics
    y_pred = model.predict(X_test_vec)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)

    # Print run summary to console
    run_name = f"Imbalance_{imbalance_method}_RandomForest_TFIDF_Trigrams"
    print("=" * 60)
    print(f"RUN: {run_name}")
    print(f"Method: {imbalance_method}")
    print("-" * 60)
    print(f"Accuracy: {accuracy:.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    print("=" * 60 + "\n")

    # Generate and save confusion matrix
    conf_matrix = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix: TF-IDF Trigrams, Imbalance={imbalance_method}")

    confusion_matrix_filename = f"confusion_matrix_{imbalance_method}.png"
    plt.savefig(confusion_matrix_filename)
    plt.close()

    # Return key stats for benchmark table tracking
    return {"method": imbalance_method, "accuracy": accuracy}


# Step 7: Run experiments for different imbalance methods
imbalance_methods = [
    "class_weights",
    "oversampling",
    "adasyn",
    "undersampling",
    "smote_enn",
]

# Accumulator list for final printing
results_log = []

for method in imbalance_methods:
    result = run_imbalanced_experiment(method)
    results_log.append(result)

# Print a final benchmark table comparing everything
print("\n##### IMBALANCE BENCHMARK SUMMARY #####")
print(f"{'Imbalance Method':<20}{'Accuracy':<10}")
print("-" * 32)
for res in results_log:
    print(f"{res['method']:<20}{res['accuracy']:.4f}")


RUN: Imbalance_class_weights_RandomForest_TFIDF_Trigrams
Method: class_weights
------------------------------------------------------------
Accuracy: 0.6756

Classification Report:
              precision    recall  f1-score   support

          -1       0.69      0.45      0.54      1650
           0       0.59      0.94      0.73      2529
           1       0.82      0.58      0.68      3154

    accuracy                           0.68      7333
   macro avg       0.70      0.66      0.65      7333
weighted avg       0.71      0.68      0.67      7333


RUN: Imbalance_oversampling_RandomForest_TFIDF_Trigrams
Method: oversampling
------------------------------------------------------------
Accuracy: 0.6778

Classification Report:
              precision    recall  f1-score   support

          -1       0.69      0.44      0.54      1650
           0       0.60      0.94      0.73      2529
           1       0.81      0.59      0.69      3154

    accuracy                           0